# 001 · Type Annotations

> **Syntax › Type Hints** · Python 3.14

Python is dynamically typed, but you can **annotate** variables, parameters and return values with types. These annotations look like C# type declarations, yet they behave very differently: the interpreter stores them and otherwise **ignores** them. A separate static checker plays the role of the C# compiler.

**In this notebook you will learn:**

- the syntax for annotating variables, parameters and return values
- that annotations are **not enforced** at runtime — and what does enforce them
- built-in generic types: `list[int]`, `dict[str, int]`, `tuple[int, ...]`, `X | None`
- how to read annotations at runtime with `__annotations__`
- Python 3.14's deferred evaluation of annotations (forward references without quotes)

## C# mental model

| C# | Python |
|---|---|
| `int count = 0;` | `count: int = 0` |
| `string Greet(string name)` | `def greet(name: str) -> str:` |
| `void Log(string msg)` | `def log(msg: str) -> None:` |
| `List<int>` | `list[int]` |
| `Dictionary<string, int>` | `dict[str, int]` |
| `(int, string)` | `tuple[int, str]` |
| `string?` (nullable reference type) | `str \| None` |
| `object` | `object` (or `Any` — see Type Hints › 003) |
| the C# compiler rejects bad types | **mypy / pyright** report them; the interpreter does not |

The key mental shift: annotations are closer to C#'s **nullable reference type annotations** than to real types — they produce *warnings from a tool*, never runtime errors.

## Annotating variables, parameters and returns

The syntax is `name: Type` for variables and parameters, and `-> Type` for the return value.

```csharp
// C#
int quantity = 3;
decimal unitPrice = 2.50m;

decimal LineTotal(int quantity, decimal unitPrice, decimal discount = 0m)
    => quantity * unitPrice - discount;
```

In [ ]:
quantity: int = 3
unit_price: float = 2.50


def line_total(quantity: int, unit_price: float, discount: float = 0.0) -> float:
    return quantity * unit_price - discount


print(line_total(quantity, unit_price))
print(line_total(2, 10.0, discount=1.5))

A function that returns nothing is annotated `-> None` (C#: `void`). You can also declare a variable's type without assigning it yet:

In [ ]:
def log(message: str) -> None:
    print(f"[log] {message}")


status: str          # declaration only — no value, no variable created yet
log("annotated!")

try:
    print(status)
except NameError as e:
    print(f"{type(e).__name__}: {e}")

## ⚠️ Annotations are NOT enforced at runtime

This is the biggest surprise for C# developers. Python happily runs code that violates its own annotations:

In [ ]:
def double(n: int) -> int:
    return n * 2


print(double(21))       # 42
print(double("ha"))     # 'haha' — str * 2 works, no error!
print(double([1]))      # [1, 1]

age: int = "forty-two"  # also fine at runtime
print(age, type(age).__name__)

If you need runtime validation (for example on data from a web request), check explicitly with `isinstance` or use a validation library. Annotations alone do nothing:

In [ ]:
def double_strict(n: int) -> int:
    if not isinstance(n, int):
        raise TypeError(f"expected int, got {type(n).__name__}")
    return n * 2


try:
    double_strict("ha")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Static type checkers are your "compiler"

The tools that *read* annotations and report errors are:

| Tool | What it is | C# analogy |
|---|---|---|
| **mypy** | the original command-line type checker | `dotnet build` type checking |
| **pyright** | fast checker from Microsoft (CLI + library) | Roslyn analyzers |
| **Pylance** | VS Code extension built on pyright — red squiggles as you type | IntelliSense / live diagnostics |

For the `double("ha")` call above, a checker reports something like:

```text
error: Argument 1 to "double" has incompatible type "str"; expected "int"
```

They are installed as dev dependencies and run in your editor or CI — not by the interpreter *(see Modules & Packages › 003 · Environments and Dependencies with uv)*.

💡 Unannotated code is treated as "anything goes" by default, so you can add annotations gradually. Most teams annotate all public function signatures, and let the checker **infer** local variable types — like using `var` in C#.

In [ ]:
def total_items(counts: dict[str, int]) -> int:
    total = 0                       # inferred as int — no annotation needed
    for name, count in counts.items():
        total += count
    return total


print(total_items({"apples": 3, "pears": 5}))

## Built-in generic types

Since Python 3.9 the built-in collection types can be subscripted directly. (Older code imports `List`, `Dict`, `Tuple` from `typing` — those still work but are deprecated aliases.)

| C# | Python annotation |
|---|---|
| `List<int>` | `list[int]` |
| `Dictionary<string, int>` | `dict[str, int]` |
| `HashSet<string>` | `set[str]` |
| `(int, string)` | `tuple[int, str]` — fixed length, per-position types |
| `int[]` / `IReadOnlyList<int>` (any length) | `tuple[int, ...]` — variable length, all `int` |
| `IEnumerable<int>` | `Iterable[int]` (from `collections.abc`) |
| `IReadOnlyDictionary<string, int>` | `Mapping[str, int]` (from `collections.abc`) |

In [ ]:
from collections.abc import Iterable, Mapping

scores: dict[str, list[int]] = {"alice": [90, 85], "bob": [70]}
point: tuple[float, float] = (1.5, -2.0)
ids: tuple[int, ...] = (4, 8, 15, 16)
tags: set[str] = {"new", "sale"}


def average(values: Iterable[float]) -> float:     # accept any iterable, like IEnumerable<T>
    items = list(values)
    return sum(items) / len(items)


def best(scores: Mapping[str, list[int]]) -> str:  # read-only view, like IReadOnlyDictionary
    return max(scores, key=lambda name: average(scores[name]))


print(average(ids), best(scores))

💡 Like in C#, accept the most **general** type you need (`Iterable`, `Mapping`, `Sequence`) and return a **concrete** type (`list`, `dict`).

## Optional values: `X | None`

C#'s nullable reference types (`string?`) map to `str | None`. The `|` union syntax works since Python 3.10; older code writes `Optional[str]` *(see Type Hints › 003 · Advanced Types)*.

```csharp
// C#
string? FindUser(int id) => id == 1 ? "alice" : null;

var name = FindUser(2);
Console.WriteLine(name?.ToUpper() ?? "<unknown>");
```

In [ ]:
def find_user(user_id: int) -> str | None:
    return "alice" if user_id == 1 else None


for uid in (1, 2):
    name = find_user(uid)
    if name is not None:          # the checker "narrows" name to str here
        print(name.upper())
    else:
        print("<unknown>")

⚠️ **Gotcha:** a default of `None` does **not** make a parameter optional in the type system automatically. Write the `| None` explicitly:

```python
def greet(name: str = None): ...         # checker error: None is not a str
def greet(name: str | None = None): ...  # correct
```

## Reading annotations at runtime: `__annotations__`

Annotations are stored as plain data on functions, classes and modules. Frameworks like `dataclasses` read them *(see Classes & OOP › 008 · Dataclasses)*.

In [ ]:
print(line_total.__annotations__)


class Product:
    name: str
    price: float
    tags: list[str]


print(Product.__annotations__)

💡 Prefer `typing.get_type_hints(obj)` or `annotationlib.get_annotations(obj)` (3.14+) over poking at `__annotations__` directly — they handle inheritance and edge cases correctly.

## Python 3.14: deferred evaluation of annotations

Before 3.14, annotations were evaluated **eagerly** when the `def` or `class` ran. Referring to a class that was not defined yet (a *forward reference*) raised `NameError`, so you had to write the type as a string: `"Node"`.

In Python **3.14** (PEP 649 / PEP 749), annotations are evaluated **lazily** — only when something asks for them. Forward references now just work without quotes, much like in C#, where declaration order never matters.

```csharp
// C# — a class can reference itself or later types freely
class Node
{
    public int Value;
    public Node? Next;
}
```

In [ ]:
class Node:
    value: int
    next: Node | None = None          # refers to Node inside its own body — no quotes needed


def make_order() -> Order:            # Order isn't defined yet — fine in 3.14
    return Order()


class Order:
    pass


print(Node.__annotations__)
print(make_order.__annotations__)

Because evaluation is deferred, an annotation that refers to a name that **never** gets defined only fails when someone asks for it. The new standard-library module `annotationlib` (3.14) lets tools read annotations in different formats without crashing:

In [ ]:
import annotationlib


def handler(event: UndefinedEvent) -> None:   # typo / missing import — no error on def
    pass


try:
    handler.__annotations__                   # evaluation happens now
except NameError as e:
    print(f"{type(e).__name__}: {e}")

print(annotationlib.get_annotations(handler, format=annotationlib.Format.STRING))
print(annotationlib.get_annotations(handler, format=annotationlib.Format.FORWARDREF))

💡 You will still see `from __future__ import annotations` and quoted annotations like `"Node"` in older code — they were the workarounds for forward references before 3.14. In 3.14-only code you don't need either.

## Summary

| Task | C# | Python |
|---|---|---|
| Typed variable | `int n = 0;` | `n: int = 0` |
| Typed function | `string F(int x)` | `def f(x: int) -> str:` |
| No return value | `void` | `-> None` |
| List / dictionary | `List<int>`, `Dictionary<string,int>` | `list[int]`, `dict[str, int]` |
| Tuple | `(int, string)` | `tuple[int, str]` |
| Variable-length tuple | `int[]` | `tuple[int, ...]` |
| Nullable | `string?` | `str \| None` |
| Who checks types | compiler (build fails) | mypy / pyright / Pylance (warnings) |
| Runtime enforcement | always | **none** — use `isinstance` if needed |
| Forward references | always allowed | allowed without quotes in 3.14 |
| Inspect at runtime | reflection | `__annotations__`, `annotationlib.get_annotations` |

## Exercises

1. Annotate this function fully: `def apply_discount(prices, percent): return [p * (1 - percent / 100) for p in prices]`. Accept any iterable of floats.
2. Write `def parse_port(text: str) -> int | None` that returns the port number, or `None` if `text` is not a number. Call it with `"8080"` and `"http"`.
3. Call your `apply_discount` with a list of **strings**. What happens at runtime, and what would a type checker say?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from collections.abc import Iterable

# 1
def apply_discount(prices: Iterable[float], percent: float) -> list[float]:
    return [p * (1 - percent / 100) for p in prices]

print(apply_discount([10.0, 20.0], 10))   # [9.0, 18.0]

# 2
def parse_port(text: str) -> int | None:
    return int(text) if text.isdigit() else None

print(parse_port("8080"), parse_port("http"))   # 8080 None

# 3 — the annotation is ignored, so the call is attempted and fails inside the
#     function with TypeError: can't multiply sequence by non-int of type 'float'.
#     A type checker would flag the call itself before you ever run it:
#     list[str] is not compatible with Iterable[float].
try:
    apply_discount(["10", "20"], 10)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")
```

</details>